In [1]:
path = "../data/"

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.memory", "4g")
    .appName("mlibs")
    .getOrCreate()
)

inDF = (
    spark.read.format("csv")
    .option("sep", ",")
    .option("inferSchema", "true")
    .option("header", "true")
    .load(path + "/US_Accidents_March23.csv")
)

inDF.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- V



###  **Zadanie 1: Analiza skumulowana (window function)**

Cel: Oblicz skumulowaną liczbę wypadków w danym stanie.


Pytanie: Które 3 stany mają największą skumulowaną liczbę wypadków?

---

In [4]:
inDF.groupBy("State").count().show()

+-----+-------+
|State|  count|
+-----+-------+
|   SC| 382557|
|   NJ| 140719|
|   DC|  18630|
|   OR| 179660|
|   VA| 303301|
|   RI|  16971|
|   KY|  32254|
|   NH|  10213|
|   MI| 162191|
|   WI|  34688|
|   CA|1741433|
|   NE|  28870|
|   CT|  71005|
|   MD| 140417|
|   DE|  14097|
|   MO|  77323|
|   IL| 168958|
|   WA| 108221|
|   IN|  67224|
|   OH| 118115|
+-----+-------+
only showing top 20 rows



---

### **Zadanie 2 – Najczęstsze warunki pogodowe**

**Cel:** Wypisz 5 najczęstszych warunków pogodowych podczas wypadków.

---

In [5]:
inDF.groupBy("Weather_Condition").count().orderBy("count", ascending=False).show(5)

+-----------------+-------+
|Weather_Condition|  count|
+-----------------+-------+
|             Fair|2560802|
|    Mostly Cloudy|1016195|
|           Cloudy| 817082|
|            Clear| 808743|
|    Partly Cloudy| 698972|
+-----------------+-------+
only showing top 5 rows




### **Zadanie 3 – Analiza godzinowa**

**Cel:** Wyciągnij godzinę z kolumny `Start_Time` i podlicz liczbę wypadków dla każdej godziny.


---

In [8]:
from pyspark.sql import functions as F

# 1. Wyciągamy godzinę z kolumny Start_Time i tworzymy nową kolumnę "Hour"
df_with_hour = inDF.withColumn("Hour", F.hour("Start_Time"))

# 2. Grupujemy po godzinie i liczymy liczbę wypadków
hourly_accidents = (
    df_with_hour.groupBy("Hour")
    .agg(F.count("ID").alias("Accident_Count"))
    .orderBy(F.desc("Accident_Count"))
)  # Sortujemy od godziny 0 do 23

# 3. Wyświetlamy wynik dla wszystkich 24 godzin
hourly_accidents.show(24)

+----+--------------+
|Hour|Accident_Count|
+----+--------------+
|   7|        587472|
|  16|        581969|
|   8|        577576|
|  17|        576015|
|  15|        525855|
|  14|        448846|
|  18|        432042|
|   6|        405837|
|  13|        396445|
|   9|        363034|
|  11|        355040|
|  12|        355001|
|  10|        342706|
|  19|        295121|
|   5|        228182|
|  20|        225226|
|  21|        191452|
|  22|        167645|
|   4|        159852|
|  23|        126539|
|   0|        112378|
|   1|         97071|
|   2|         93073|
|   3|         84017|
+----+--------------+





### **Zadanie 4 – Dzień vs. Noc**

**Cel:** Policz liczbę wypadków, które wydarzyły się w ciągu dnia i nocy (`Sunrise_Sunset`).

---



In [11]:
Sunrise_Sunset_accidents = (
    inDF.groupBy("Sunrise_Sunset")
    .agg(F.count("ID").alias("Sunrise_Sunset_Count"))
    .orderBy(F.desc("Sunrise_Sunset_Count"))
)

# 3. Wyświetlamy wynik dla wszystkich 24 godzin
Sunrise_Sunset_accidents.show(2)

+--------------+--------------------+
|Sunrise_Sunset|Sunrise_Sunset_Count|
+--------------+--------------------+
|           Day|             5334553|
|         Night|             2370595|
+--------------+--------------------+
only showing top 2 rows




### **Zadanie 5 – Filtrowanie poważnych wypadków przy złej pogodzie**

**Cel:** Wybierz tylko wypadki o powadze ≥ 4 i złych warunkach pogodowych.

---


In [14]:
# 1. Definiujemy, jakie słowa kluczowe w kolumnie Weather_Condition oznaczają "złą pogodę"
# Użyjemy funkcji .rlike() z wyrażeniem regularnym, żeby złapać np. "Heavy Rain", "Light Snow", "Fog" itp.
zla_pogoda_filter = F.col("Weather_Condition").rlike(
    "(?i)(rain|snow|fog|storm|ice|sleet|thunderstorm)"
)

# 2. Filtrujemy dane: Severity >= 4 ORAZ zła pogoda
# Pamiętaj: w PySparku przy łączeniu warunków zawsze bierzemy je w nawiasy (...)
powazne_wypadki_zla_pogoda = (
    inDF.filter((F.col("Severity") >= 4) & zla_pogoda_filter)
    .groupBy("Weather_Condition")
    .agg(F.count("ID").alias("count"))
    .orderBy(F.desc("count"))
)

powazne_wypadki_zla_pogoda.show(10)

+--------------------+-----+
|   Weather_Condition|count|
+--------------------+-----+
|          Light Rain| 9360|
|          Light Snow| 4759|
|                 Fog| 2774|
|                Rain| 2029|
|          Heavy Rain|  753|
|                Snow|  718|
|             T-Storm|  409|
|          Heavy Snow|  263|
|Light Rain with T...|  249|
|  Light Snow / Windy|  210|
+--------------------+-----+
only showing top 10 rows



### **Zadanie 6 – Kierunki wiatru**

**Cel:** Znajdź najczęściej występujące kierunki wiatru podczas wypadków.

---

In [15]:
Sunrise_Sunset_accidents = (
    inDF.groupBy("Wind_Direction")
    .agg(F.count("ID").alias("Wind_Direction_Count"))
    .orderBy(F.desc("Wind_Direction_Count"))
)

Sunrise_Sunset_accidents.show(2)

+--------------+--------------------+
|Wind_Direction|Wind_Direction_Count|
+--------------+--------------------+
|          CALM|              961624|
|             S|              419989|
+--------------+--------------------+
only showing top 2 rows



---

## Zadanie 7 `groupBy` + `agg`

**Zadanie:**
Policz średnią widoczność (`Visibility(mi)`) i średnią temperaturę (`Temperature(F)`) dla każdego stanu (`State`). Posortuj wyniki malejąco po średniej temperaturze.


In [16]:
# 1. Grupujemy po kolumnie "State"
state_weather_stats = (
    inDF.groupBy("State")
    .agg(
        F.avg("Visibility(mi)").alias("Average_Visibility"),
        F.avg("Temperature(F)").alias("Average_Temperature"),
    )
    .orderBy(F.desc("Average_Temperature"))
)  # 2. Sortujemy malejąco po średniej temperaturze

# 3. Wyświetlamy wyniki w ładnej dla oka tabeli Pandas
state_weather_stats.show(10)

+-----+------------------+-------------------+
|State|Average_Visibility|Average_Temperature|
+-----+------------------+-------------------+
|   FL| 9.512227927507954|  75.37415001085002|
|   AZ| 10.21233233130719|  72.65462110278449|
|   LA| 9.162557622321065|  70.03253580137145|
|   TX|  9.24599442163708|  69.87033370705942|
|   AL| 9.094627564160378|   66.4881669742438|
|   MS| 8.992551115862621|  66.13484858613545|
|   SC| 9.076028494389934|  64.52727577571342|
|   GA| 8.980158840900478|  64.29435559623303|
|   CA| 9.089183886750666| 63.900452383536674|
|   OK| 9.397583543524696|  63.80269713045579|
+-----+------------------+-------------------+
only showing top 10 rows



---

## Zadanie 8 `orderBy`

**Zadanie:**
Znajdź 5 wypadków o najdłuższym czasie trwania (`End_Time - Start_Time`). Wyświetl ich `ID`, `City`, `State` i czas trwania w minutach.

---

In [17]:
df_with_duration = inDF.withColumn(
    "Duration_Minutes", (F.unix_timestamp("End_Time") - F.unix_timestamp("Start_Time")) / 60
)

# 2. Wybieramy tylko interesujące nas kolumny, sortujemy malejąco i bierzemy 5 pierwszych
top_5_longest_accidents = df_with_duration.select(
    "ID", "City", "State", "Duration_Minutes"
).orderBy(F.desc("Duration_Minutes"))

# 3. Wyświetlamy wynik w ładnej tabeli Pandas
top_5_longest_accidents.show(5)

+---------+-----------+-----+-----------------+
|       ID|       City|State| Duration_Minutes|
+---------+-----------+-----+-----------------+
|A-4810425| Wilmington|   DE|        2812999.0|
|A-5053641| Wilmington|   DE|        2812999.0|
|A-5399002|Southampton|   NY|       2236405.75|
|A-4014778|Southampton|   NY|2236355.533333333|
|A-5073315|Southampton|   NY|2236355.533333333|
+---------+-----------+-----+-----------------+
only showing top 5 rows




## Zadanie 9 `to_timestamp` i różnice czasowe

**Zadanie:**
Oblicz średnią długość wypadków (w minutach) w dzień (`Sunrise_Sunset = 'Day'`) i w nocy (`Sunrise_Sunset = 'Night'`).

---

In [18]:
df_with_duration = inDF.withColumn(
    "Duration_Minutes", (F.unix_timestamp("End_Time") - F.unix_timestamp("Start_Time")) / 60
)

# 2. Wybieramy tylko interesujące nas kolumny, sortujemy malejąco i bierzemy 5 pierwszych
top_5_longest_accidents = df_with_duration.groupBy("Sunrise_Sunset").avg("Duration_Minutes")

# 3. Wyświetlamy wynik w ładnej tabeli Pandas
top_5_longest_accidents.show()

+--------------+---------------------+
|Sunrise_Sunset|avg(Duration_Minutes)|
+--------------+---------------------+
|          NULL|   3284.4110348733852|
|         Night|     562.145285269462|
|           Day|    379.7371654413549|
+--------------+---------------------+





## Zadanie 10 `filter` + `isin`

**Zadanie:**
Znajdź wszystkie wypadki, które wydarzyły się w stanie **California (CA)** lub **Texas (TX)** **w złych warunkach pogodowych** (`Weather_Condition IN ('Rain', 'Snow', 'Fog')`).

---

In [19]:
# 1. Definiujemy listy interesujących nas wartości
wybrane_stany = ["CA", "TX"]
zla_pogoda = ["Rain", "Snow", "Fog"]

# 2. Filtrujemy dane przy użyciu metody .isin()
# Pamiętaj o nawiasach dla każdego z warunków!
wypadki_ca_tx_zla_pogoda = inDF.filter(
    (F.col("State").isin(wybrane_stany)) & (F.col("Weather_Condition").isin(zla_pogoda))
)

# 3. Wybieramy kluczowe kolumny i wyświetlamy 10 przykładowych rekordów
wypadki_ca_tx_zla_pogoda.select("ID", "State", "Weather_Condition", "City", "Start_Time").show(10)

+------+-----+-----------------+---------------+-------------------+
|    ID|State|Weather_Condition|           City|         Start_Time|
+------+-----+-----------------+---------------+-------------------+
|A-4382|   CA|              Fog|     Santa Cruz|2016-07-27 05:07:44|
|A-5598|   CA|              Fog|American Canyon|2016-12-05 01:28:52|
|A-5608|   CA|              Fog|        Lathrop|2016-12-05 07:40:39|
|A-5610|   CA|              Fog|          Tracy|2016-12-05 08:16:50|
|A-5795|   CA|              Fog|     Sacramento|2016-12-06 09:14:25|
|A-5806|   CA|              Fog|     Sacramento|2016-12-06 09:54:21|
|A-5812|   CA|              Fog|     Sacramento|2016-12-06 10:21:40|
|A-6065|   CA|             Rain|     Santa Rosa|2016-12-07 19:43:06|
|A-6070|   CA|             Rain|     Santa Rosa|2016-12-07 19:55:08|
|A-6088|   CA|             Rain|     Lower Lake|2016-12-07 20:50:43|
+------+-----+-----------------+---------------+-------------------+
only showing top 10 rows




## Zadanie 11 `withColumn`

**Zadanie:**
Dodaj nową kolumnę `is_long`, która przyjmuje wartość:

* `1` jeśli wypadek trwał dłużej niż 60 minut,
* `0` w przeciwnym razie.

---


In [20]:
df_duration = inDF.withColumn(
    "Duration_Minutes", (F.unix_timestamp("End_Time") - F.unix_timestamp("Start_Time")) / 60
)

# 2. Dodajemy kolumnę 'is_long' za pomocą F.when()
# F.when(warunek, wartość_jeśli_prawda).otherwise(wartość_jeśli_fałsz)
df_with_is_long = df_duration.withColumn(
    "is_long", F.when(F.col("Duration_Minutes") > 60, 1).otherwise(0)
)

# 3. Wyświetlamy wynik (wybieramy kolumny czasu, żebyś widział czy warunek zadziałał poprawnie)
df_with_is_long.select("ID", "Start_Time", "End_Time", "Duration_Minutes", "is_long").show(10)

+----+-------------------+-------------------+----------------+-------+
|  ID|         Start_Time|           End_Time|Duration_Minutes|is_long|
+----+-------------------+-------------------+----------------+-------+
| A-1|2016-02-08 05:46:00|2016-02-08 11:00:00|           314.0|      1|
| A-2|2016-02-08 06:07:59|2016-02-08 06:37:59|            30.0|      0|
| A-3|2016-02-08 06:49:27|2016-02-08 07:19:27|            30.0|      0|
| A-4|2016-02-08 07:23:34|2016-02-08 07:53:34|            30.0|      0|
| A-5|2016-02-08 07:39:07|2016-02-08 08:09:07|            30.0|      0|
| A-6|2016-02-08 07:44:26|2016-02-08 08:14:26|            30.0|      0|
| A-7|2016-02-08 07:59:35|2016-02-08 08:29:35|            30.0|      0|
| A-8|2016-02-08 07:59:58|2016-02-08 08:29:58|            30.0|      0|
| A-9|2016-02-08 08:00:40|2016-02-08 08:30:40|            30.0|      0|
|A-10|2016-02-08 08:10:04|2016-02-08 08:40:04|            30.0|      0|
+----+-------------------+-------------------+----------------+-


## Zadanie 12 `count`

**Zadanie:**
Policz liczbę różnych miast (`DISTINCT City`) w każdym stanie (`State`). Posortuj wyniki malejąco po liczbie unikalnych miast.

---

In [21]:
# 1. Grupujemy po stanie i liczymy unikalne miasta za pomocą count_distinct
unique_cities_per_state = (
    inDF.groupBy("State")
    .agg(F.count_distinct("City").alias("Unique_Cities_Count"))
    .orderBy(F.desc("Unique_Cities_Count"))
)  # 2. Sortujemy wyniki malejąco

# 3. Wyświetlamy top 10 stanów z największą liczbą unikalnych miast
unique_cities_per_state.show(10)

+-----+-------------------+
|State|Unique_Cities_Count|
+-----+-------------------+
|   PA|               1422|
|   CA|               1268|
|   NY|               1215|
|   TX|                883|
|   IL|                785|
|   OH|                780|
|   VA|                660|
|   MN|                656|
|   NC|                612|
|   MI|                590|
+-----+-------------------+
only showing top 10 rows
